# Redução de Dimensionalidade com PCA (`housing.csv`)

Projeto prático aplicando **Análise de Componentes Principais (PCA)** para reduzir a dimensionalidade do conjunto de dados imobiliários, visualizar os dados transformados e avaliar o erro de reconstrução.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

## 1. Carregamento e Inspeção dos Dados

In [2]:
df = pd.read_csv('housing.csv')
print('Dimensões:', df.shape)
display(df.head())
df.info()

## 2. Pré-processamento e Padronização

In [3]:
df_proc = df.copy()

binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_proc[col] = df_proc[col].map({'yes': 1, 'no': 0})

furnishing_map = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
df_proc['furnishingstatus'] = df_proc['furnishingstatus'].map(furnishing_map)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_proc)

## 3. Aplicação do PCA e Erro de Reconstrução

In [4]:
pca_full = PCA()
pca_full.fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

n_80 = int(np.argmax(cum_var >= 0.80) + 1)
n_90 = int(np.argmax(cum_var >= 0.90) + 1)
print(f'Componentes para >= 80% da variância: {n_80} ({cum_var[n_80 - 1]:.2%})')
print(f'Componentes para >= 90% da variância: {n_90} ({cum_var[n_90 - 1]:.2%})')

# Redução para 2 componentes (visualização)
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)
mse_2 = mean_squared_error(X_scaled, pca_2.inverse_transform(X_pca_2))

# Redução com >= 90% da variância
pca_90 = PCA(n_components=n_90)
X_pca_90 = pca_90.fit_transform(X_scaled)
mse_90 = mean_squared_error(X_scaled, pca_90.inverse_transform(X_pca_90))

print(f'Variância explicada (2 componentes): {cum_var[1]:.2%}')
print(f'Erro de reconstrução (MSE - 2 comps): {mse_2:.4f}')
print(f'Erro de reconstrução (MSE - {n_90} comps): {mse_90:.4f}')

## 4. Visualização e Interpretação dos Componentes

In [5]:
loadings = pd.DataFrame(pca_2.components_.T, index=df_proc.columns, columns=['PC1', 'PC2'])
display(loadings.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(cum_var) + 1), pca_full.explained_variance_ratio_ * 100, alpha=0.55, color='#6366f1', label='Individual')
axes[0].plot(range(1, len(cum_var) + 1), cum_var * 100, marker='o', color='#1e1b4b', label='Acumulada')
axes[0].axhline(80, color='#059669', linestyle='--', linewidth=1, label='80%')
axes[0].axhline(90, color='#dc2626', linestyle='--', linewidth=1, label='90%')
axes[0].set_title('Variância Explicada por Componente')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.25)

sc = axes[1].scatter(X_pca_2[:, 0], X_pca_2[:, 1], c=df['price'] / 1e6, cmap='viridis', alpha=0.8, s=30)
plt.colorbar(sc, ax=axes[1], label='Preço (Milhões)')
axes[1].set_title('Projeção PCA 2D (PC1 vs PC2)')
axes[1].set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]:.1%})')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 5. Interpretação dos resultados

- **Variância:** 2 componentes explicam 36,44% da variância (PC1 24,62%, PC2 11,81%); 8 componentes chegam a 80,65% e 10 a 90,55%. A informação está espalhada, então a projeção 2D é útil para visualizar, mas perde muita informação.
- **Erro de reconstrução (MSE na escala padronizada):** 0,6356 com 2 componentes e 0,0945 com 10 componentes.
- **PC1 (porte e valor do imóvel):** maiores pesos em `price` (0,50), `area` (0,34), `bathrooms` (0,33) e `airconditioning` (0,29): separa imóveis maiores e mais caros dos compactos.
- **PC2 (configuração interna vs. infraestrutura externa):** positivo em `stories` (0,49) e `bedrooms` (0,34), negativo em `basement` (-0,48), `guestroom` (-0,36) e `prefarea` (-0,35).
- **Limitações:** `furnishingstatus` foi codificada de forma ordinal e as binárias padronizadas, o que é uma simplificação; o PCA é linear e `price` entrou como atributo na decomposição.